# Concepts — the building blocks of an AI agent

Before we build the real project (an **Inbox Agent**), let's learn its three building blocks one at a time, using the **OpenAI Agents SDK**.

1. **An agent** — a model + instructions you can run.
2. **A tool** — a normal Python function the agent can choose to call.
3. **Structured output** — forcing the agent to answer in an exact shape.

That's everything the Inbox Agent is made of.

> Run each cell with **Shift + Enter**. Make sure you selected the `.venv` kernel (top right).

## 1. The smallest possible agent

An agent needs three things: a **name**, **instructions**, and a **model**. Then we `run` it.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner

# Loads OPENAI_API_KEY from your .env file
load_dotenv(override=True)

agent = Agent(
    name="Helper",
    instructions="You are a helpful assistant.",
    model="gpt-4o-mini",
)

result = await Runner.run(agent, "Say hello in one short sentence.")
print(result.final_output)

That's it — that is a working agent. Now let's give one a **tool**.

## 2. Giving the agent a tool

A tool is just a Python function with the `@function_tool` decorator on top. The agent decides on its own when to call it — we don't call it ourselves.

In [ ]:
from agents import function_tool

@function_tool
def get_word_count(text: str) -> int:
    """Return how many words are in the given text."""
    return len(text.split())

tool_agent = Agent(
    name="Counter",
    instructions="When asked about word counts, use your tools to answer.",
    model="gpt-4o-mini",
    tools=[get_word_count],
)

result = await Runner.run(tool_agent, "How many words are in: the quick brown fox jumps")
print(result.final_output)

The agent read your question, decided to call `get_word_count`, and used the result to answer. **That** is what makes it an agent and not just a chatbot.

In our project, the tool will be `read_emails()` instead of counting words.

## 3. Structured output

Sometimes we don't want free text — we want clean data we can build on. We describe the shape we want with a **Pydantic** model and pass it as `output_type`. The agent is then forced to fill it in.

In [ ]:
from pydantic import BaseModel, Field

class Classification(BaseModel):
    category: str = Field(description="One of: question, request, spam, notification")
    needs_reply: bool = Field(description="Does this message need a reply?")

classifier = Agent(
    name="Classifier",
    instructions="Classify the message the user gives you.",
    model="gpt-4o-mini",
    output_type=Classification,
)

result = await Runner.run(classifier, "Hey, can you send me the invoice for last month?")
print(result.final_output)
print("Category:", result.final_output.category)
print("Needs reply:", result.final_output.needs_reply)

Notice we get back real fields (`category`, `needs_reply`) — not a paragraph. That's what lets us build a GUI on top of the agent's answer.

## Putting it together → the project

The **Inbox Agent** uses exactly these three ideas at once:
- a **tool** `read_emails()` to fetch the inbox,
- **structured output** to say, per email, *needs reply / why / suggested draft*,
- and a **human-in-the-loop** step so you approve before anything is sent.

Now open **`project/inbox_agent.py`** — and we'll build it together, then wrap it in a clickable GUI with **`project/app.py`**.